# Questions

A question is the basic building block of evaluation in Karenina. Each question carries the text sent to an LLM, a reference answer that anchors the evaluation, and intrinsic metadata (author, sources, timestamps, keywords). The `Question` model is self-contained: everything that describes the question itself lives on the object.

In [ ]:
# Mock cell: ensures examples execute without live API keys.
# This cell is hidden in rendered documentation.
import hashlib
from unittest.mock import MagicMock, patch

mock_modules = {}
for mod in ["sqlalchemy", "sqlalchemy.orm", "sqlalchemy.ext", "sqlalchemy.ext.declarative"]:
    mock_modules[mod] = MagicMock()

with patch.dict("sys.modules", mock_modules):
    from pydantic import Field

    from karenina.benchmark import Benchmark
    from karenina.schemas.entities import BaseAnswer
    from karenina.schemas.entities.question import Question

In [ ]:
# Standard imports for working with questions
from karenina.benchmark import Benchmark
from karenina.schemas.entities import BaseAnswer
from karenina.schemas.entities.question import Question

## What Is a Question?

The `Question` model carries core evaluation data and intrinsic metadata:

**Core fields:**

| Field | Type | Required | Description |
|-------|------|----------|-------------|
| `question` | `str` | Yes | The question text sent to the LLM (minimum 1 character) |
| `raw_answer` | `str` | Yes | The human-readable reference answer (minimum 1 character) |
| `keywords` | `list[str]` | No | Keywords for filtering and organization (default: empty list) |
| `few_shot_examples` | `list[dict[str, str]] \| None` | No | Example question-answer pairs for parsing guidance |
| `id` | `str` (computed) | Auto | Deterministic MD5 hash of the question text |

**Intrinsic metadata** (carried on the Question object itself):

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `date_created` | `str` | Now (ISO) | When the question was created |
| `date_modified` | `str` | Now (ISO) | When the question was last modified |
| `answer_template` | `str \| None` | `None` | Template code attached to this question |
| `author` | `dict \| None` | `None` | Author information (name, affiliation, etc.) |
| `sources` | `list[dict] \| None` | `None` | Source documents or references |
| `custom_metadata` | `dict \| None` | `None` | Arbitrary key-value pairs |
| `question_rubric` | `dict \| None` | `None` | Question-specific rubric traits |

### What the Question Text Does in the Pipeline

The `question` field is the prompt submitted to the answering LLM during the [verification pipeline](../verification-pipeline.md). By default, karenina sends it as a bare user message with no wrapping or hidden instructions.

Two optional configurations add context around it: (1) a system prompt can be set on the answering [`ModelConfig`](../../reference/configuration/model-config.md), sent as a separate system message; (2) [few-shot examples](../few-shot.md), if enabled, are prepended in a `"Question: ...\nAnswer: ..."` format. Beyond these, nothing is injected. If the author needs the LLM to have background information, domain context, or task instructions, all of that must be included in the question text itself (or in a configured system prompt).

The same text also appears in other pipeline stages: the [Judge LLM](../verification-pipeline.md) receives it as context during parsing, and [rubric evaluators](../rubrics/index.md) receive it when assessing response quality. Unlike the answering stage, the parsing and rubric stages have their own system prompts and [instruction builders](../prompt-assembly.md) assembled by the framework. The answering stage is deliberately bare: the benchmark author controls exactly what the model sees.

In [ ]:
q = Question(
    question="What is the putative target of venetoclax?",
    raw_answer="BCL2 (B-cell lymphoma 2)",
    keywords=["pharmacology", "oncology"],
)

print(f"Question:   {q.question}")
print(f"Raw answer: {q.raw_answer}")
print(f"Keywords:   {q.keywords}")
print(f"ID:         {q.id}")

### What `raw_answer` Does (and Does Not Do)

`raw_answer` is metadata for the benchmark author and downstream tooling. It is never sent to the answering LLM and never sent to the Judge LLM.

Its primary programmatic use is as input to the automatic template generator (`karenina.benchmark.authoring.answers.generator`), which reads the question and `raw_answer` to derive a structured answer template with ground truth values and verification logic.

It is also stored in verification results and exported in reports (CSV, markdown), so reviewers can see at a glance what the expected answer was.

The template's [`ground_truth()` method](../answer-templates.md), not `raw_answer`, is what actually controls verification. `raw_answer` is the human-readable source from which the template author (or the generator) derives the programmatic answer key.

<div class="admonition note">
<p class="admonition-title">Backward compatibility: <code>tags</code> to <code>keywords</code></p>
<p>The field formerly named <code>tags</code> has been renamed to <code>keywords</code>. A model validator accepts the legacy <code>tags</code> key during construction and automatically converts it to <code>keywords</code>, so existing checkpoint files and code using <code>tags=</code> continue to work.</p>
</div>

## Deterministic IDs

Every question gets an ID computed as an MD5 hash of its text. The same question text always produces the same ID, which enables reliable cross-referencing between benchmarks, results, and traces.

In [ ]:
# The ID is deterministic: same text produces same ID
q1 = Question(question="What is the capital of France?", raw_answer="Paris")
q2 = Question(question="What is the capital of France?", raw_answer="Paris")
print(f"q1.id: {q1.id}")
print(f"q2.id: {q2.id}")
print(f"Same:  {q1.id == q2.id}")

In [ ]:
# Under the hood: MD5 of the question text
manual_id = hashlib.md5(b"What is the capital of France?").hexdigest()
print(f"Manual hash: {manual_id}")
print(f"Matches:     {manual_id == q1.id}")

You can override the auto-generated ID by passing `question_id` when adding to a benchmark:

In [ ]:
benchmark = Benchmark.create(name="ID Demo", version="0.1.0")
custom_id = benchmark.add_question(
    question="What is the capital of France?",
    raw_answer="Paris",
    question_id="france-capital-001",
)
print(f"Custom ID: {custom_id}")

<div class="admonition warning">
<p class="admonition-title">Changing text changes the ID</p>
<p>If you modify a question's text (even whitespace or capitalization), the auto-generated ID changes. This breaks cross-references to results and traces tied to the old ID. If you need to edit question text while preserving the ID, use a custom <code>question_id</code>.</p>
</div>

## `raw_answer` vs Template `ground_truth`

These two concepts are related but serve different purposes:

| Concept | Where It Lives | Who Sees It | Purpose |
|---------|---------------|-------------|---------|
| `raw_answer` | `Question.raw_answer` | Benchmark authors; stored in results for reference | Human-readable reference answer written in plain language |
| `self.correct` | Set inside `ground_truth()` method | Only `verify()` | Programmatic answer key used by the `verify()` method to check correctness |

The relationship: `raw_answer` is the source of truth from which the template author derives `self.correct`. A well-written `raw_answer` describes the expected answer in plain language. The `ground_truth()` method translates that into structured values that `verify()` can compare programmatically.

`raw_answer` is **not** sent to the [Judge LLM](../verification-pipeline.md) during parsing. The Judge receives only the question text, the LLM's response, and the template's JSON schema. The `raw_answer` serves as the author's reference when writing the template's `ground_truth()` method, and it is stored in verification results for human review.

In [ ]:
# The raw_answer describes the expected answer in natural language
q = Question(
    question="What is the putative target of venetoclax?",
    raw_answer="BCL2 (B-cell lymphoma 2), an anti-apoptotic protein",
)


# The template's ground_truth() derives structured values from that
# (see Answer Templates for full details on BaseAnswer)
class Answer(BaseAnswer):
    target: str = Field(
        description=(
            "The direct protein target of venetoclax as stated in the response. "
            "Extract the protein name or gene symbol (e.g., 'BCL2', 'Bcl-2', "
            "'B-cell lymphoma 2'). If the response mentions multiple proteins, "
            "extract only the one identified as the direct pharmacological "
            "target, not downstream effectors or pathway members."
        )
    )

    def ground_truth(self):
        # Derived from raw_answer: "BCL2 (B-cell lymphoma 2)"
        self.correct = {"target": "BCL2"}

    def verify(self) -> bool:
        return self.target.strip().upper().replace("-", "") == self.correct["target"]

<div class="admonition tip">
<p class="admonition-title">Writing a good <code>raw_answer</code></p>
<p>Although <code>raw_answer</code> is not sent to the Judge LLM, it still matters. It is the reference you (the benchmark author) use when writing the template's <code>ground_truth()</code> method. A specific <code>raw_answer</code> like <code>"BCL2 (B-cell lymphoma 2)"</code> makes it clear what <code>self.correct</code> should contain, while a vague one like <code>"yes"</code> leaves the template author guessing. It also appears in verification results, so reviewers can see at a glance what the expected answer was.</p>
</div>

A poorly written `raw_answer` has practical consequences: the automatic template generator produces weaker templates because it has less signal to extract ground truth from, and reviewers scanning results cannot quickly tell whether a failure is a real model mistake or an ambiguous expectation.

## Keywords

Keywords are pure metadata for filtering and organization. They never reach any LLM (answering, judge, or rubric evaluator) and have no effect on pipeline execution. Their value is in managing benchmarks at scale: filtering with `search_questions()`, grouping results by topic during analysis, and slicing exports by domain or difficulty. For operational examples, see [Benchmark Operations](../../workflows/creating-benchmarks/benchmark-operations.md#filtering-and-search).

## Metadata Fields

`author`, `sources`, and `custom_metadata` are provenance and tracking metadata. They are never used in pipeline execution.

`author` records who wrote the question (name, affiliation). `sources` records where the question or expected answer came from (papers, databases, textbooks). `custom_metadata` is an open dictionary for any domain-specific attributes (difficulty, regulatory category, review status).

All three are preserved in [checkpoint files](../checkpoints.md), stored in the database, and included in exports. They are useful for audit trails, filtering during analysis, and compliance workflows that require provenance tracking.

## Few-Shot Examples

Questions can carry example question-answer pairs. When few-shot examples are enabled in `VerificationConfig`, they are prepended to the question text before it is sent to the answering LLM. Each example is rendered as `"Question: ...\nAnswer: ..."`, followed by the actual question in the same format. They are not sent to the Judge LLM during parsing. The primary use case is guiding the answering model's response format or demonstrating the expected level of detail. For configuration modes and best practices, see [Few-Shot](../few-shot.md).

## The `finished` Flag

The `finished` flag determines whether a question enters the verification pipeline. It is tracked separately from the question's intrinsic data because `finished` is a property of the question's membership in a benchmark, not an intrinsic property of the question itself. A question can be finished in one benchmark and unfinished in another.

The default depends on the interface:

- **Python API** (`add_question()`): defaults to `finished=True`, because questions added programmatically are assumed to be complete
- **GUI** (karenina-gui): always passes `finished=False`, prompting the user to review before verification

Only finished questions are included when the verification pipeline runs. The most common pipeline troubleshooting issue is an empty result set from `run_verification()`. In nearly all cases, this means every question is marked `finished=False`. This is especially common when questions were created in the GUI (which defaults to `finished=False`) and then run via the Python API without explicitly marking them finished first.

For managing finished status (marking, toggling, batch operations), see [Benchmark Operations](../../workflows/creating-benchmarks/benchmark-operations.md#managing-question-state).

## Next Steps

- [Benchmarks](benchmarks.md): the benchmark as a package, metadata, persistence
- [Answer Templates](../answer-templates.md): how to write templates, field types, `ground_truth()` and `verify()`
- [Evaluation Modes](../evaluation-modes.md): how `finished` status, templates, and rubrics determine what the pipeline runs
- [Benchmark Operations](../../workflows/creating-benchmarks/benchmark-operations.md): adding questions, managing finished status, accessing question data, writing good `raw_answer` values
- [Creating Benchmarks](../../workflows/creating-benchmarks/index.md): step-by-step authoring workflow
- [Checkpoints](../checkpoints.md): how benchmarks persist as JSON-LD files
- [Few-Shot](../few-shot.md): configuring example injection for parsing accuracy